# Módulo AI3 · RAG y agentes — Práctica guiada

**Producto:** *asistente de soporte de TiendaSur* con conocimiento propio, y **medido**.

1. **Corpus y evaluación primero:** un manual de atención al cliente y 36 preguntas con la respuesta marcada.
2. **Pipeline de recuperación:** chunking → embeddings locales → **base vectorial real** (SQLite + `sqlite-vec`) → **BM25** → **fusión híbrida RRF** → **reranking**.
3. **Comparación con métricas** (acierto@k y MRR) de cada variante.
4. **Generación con citas** y **agente** con herramientas (se mantienen del módulo anterior, ahora sobre la búsqueda mejorada).

> Todo corre en CPU, sin descargas y sin clave. Las celdas marcadas **[API]** usan Claude si hay `ANTHROPIC_API_KEY`; sin ella siguen con un camino simulado.

In [ ]:
import json
import math
import os
import re
import sqlite3
import textwrap
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

try:                                   # en Google Colab la clave vive en los "secretos"
    from google.colab import userdata
    os.environ.setdefault("ANTHROPIC_API_KEY", userdata.get("ANTHROPIC_API_KEY") or "")
except ImportError:
    pass

try:
    import sqlite_vec                  # pip install sqlite-vec  (en Colab: %pip install sqlite-vec)
except ImportError:
    sqlite_vec = None
    print("Falta sqlite-vec: ejecuta  pip install sqlite-vec")

TIENE_CLAVE = bool(os.environ.get("ANTHROPIC_API_KEY"))
MODELO = "claude-opus-5-5"
CARPETA = Path("proyecto-rag")        # aquí se guarda la base de datos (carpeta ignorada por git)
CARPETA.mkdir(exist_ok=True)
print("clave de API presente:", TIENE_CLAVE, "| sqlite-vec:", getattr(sqlite_vec, "__version__", None),
      "| SQLite", sqlite3.sqlite_version)

## 3.1 · El corpus y la evaluación (antes que el código)

Un RAG sin evaluación es una opinión. Empezamos fijando **qué es acertar**: cada pregunta lleva la frase del manual que contiene la respuesta, y un chunk es **relevante** si la contiene. Así podemos comparar estrategias de chunking distintas con el mismo examen.

Las preguntas están redactadas **como las escribiría un cliente** (con palabras distintas de las del manual: «anular» frente a «cancelar», «cupones» frente a «códigos»).

In [ ]:
MANUAL = """# Manual de atención al cliente de TiendaSur

## Envíos

### Plazos
El envío estándar tarda de 3 a 5 días laborables en la península. Los pedidos urgentes llegan al día siguiente si se confirman antes de las 14:00. A Baleares y Canarias el plazo es de 5 a 8 días laborables y en Canarias puede haber trámites de aduana.

### Costes
El envío es gratuito a partir de 39 EUR de compra en la península. Por debajo de ese importe cuesta 4,90 EUR. El envío urgente cuesta 8,50 EUR siempre.

### Seguimiento
Al salir del almacén recibirás un correo con el enlace de seguimiento del transportista. Si el estado no cambia en 72 horas, abre una incidencia desde tu cuenta.

## Facturas

### Emisión
La factura se emite el mismo día de la compra y se envía por correo en PDF. Puedes descargarla de nuevo desde "Mis pedidos".

### Factura con CIF
Para una factura a nombre de empresa hay que indicar el CIF antes de pagar. Si se pide después, se emite en 24 horas con datos fiscales completos, pero solo hasta 30 días tras la compra.

### Rectificativas
Si hay un error en la factura se emite una rectificativa; nunca se modifica la original. La rectificativa tarda hasta 5 días laborables.

## Devoluciones

### Plazo y condiciones
Hay 30 días naturales para devolver. El producto debe estar sin usar y en su embalaje original. El café abierto no se admite por motivos de higiene.

### Coste de la devolución
Los gastos de envío de la devolución corren a cargo del cliente, salvo que el producto sea defectuoso o se haya enviado por error.

### Reembolso
El dinero se devuelve por el mismo medio de pago en un máximo de 10 días laborables desde que el almacén recibe el paquete. Los pagos con transferencia se reembolsan a la cuenta bancaria indicada.

## Productos dañados o perdidos

### Paquete dañado
Si el paquete llega dañado, abre un ticket con foto en menos de 48 horas. Con la foto se repone el producto sin coste. Las tazas rotas en el transporte se reenvían en 3 días.

### Pedido que no llega
Si el pedido no llega en 7 días laborables, contacta con soporte: se reenvía gratis o se reembolsa, a tu elección.

## Café

### Frescura
Tostamos bajo pedido cada martes y jueves. El café sale como máximo 48 horas después del tostado. Consúmelo en las seis semanas siguientes para disfrutar de todo su aroma.

### Molienda
Puedes pedir el café en grano o molido. Ofrecemos tres moliendas: fina para espresso, media para cafetera de goteo y gruesa para prensa francesa. La molienda no se puede cambiar una vez preparado el pedido.

### Conservación
Guarda el café en un recipiente hermético, lejos de la luz y del calor. No lo metas en la nevera: absorbe humedad y olores.

### Alérgenos
Nuestras instalaciones procesan frutos secos y gluten; los cafés aromatizados pueden contener trazas. El café puro no contiene alérgenos declarables.

## Suscripciones

### Cómo funcionan
La suscripción de café envía una bolsa de 250 g cada 2 o 4 semanas, con un 10 % de descuento. Se cobra 3 días antes de cada envío.

### Pausar o cancelar
Puedes pausar o cancelar desde tu cuenta hasta 48 horas antes del cobro, sin penalización. Si el cobro ya se ha hecho, el envío sale igualmente.

## Pagos y descuentos

### Medios de pago
Aceptamos tarjeta, Bizum, PayPal y transferencia. Las transferencias deben recibirse en 3 días o el pedido se cancela automáticamente.

### Códigos de descuento
Los códigos no son acumulables. El código BIENVENIDA10 da un 10 % en la primera compra. El código se introduce en la cesta antes de pagar y no se puede aplicar después.

### Programa de puntos
Cada euro gastado suma un punto. 500 puntos equivalen a 5 EUR de descuento. Los puntos caducan a los 12 meses.

## Menaje

### Garantía
El menaje tiene 3 años de garantía legal por defectos de fabricación. La garantía no cubre golpes ni el uso en lavavajillas de las piezas de cerámica artesanal.

### Cuidado de la cerámica
Lava a mano las piezas de cerámica artesanal con agua tibia. Evita los cambios bruscos de temperatura para que el esmalte no se agriete.

## Tienda física

### Horario y recogida
La tienda de Sevilla abre de lunes a sábado de 10:00 a 20:00. Los pedidos online pueden recogerse allí gratis en 24 horas, con el número de pedido y un documento de identidad.

## Datos personales

### Qué guardamos
Guardamos nombre, dirección, correo y el historial de pedidos para gestionar compras y facturas. No almacenamos el número completo de la tarjeta.

### Tus derechos
Puedes pedir acceso, rectificación o borrado de tus datos escribiendo a privacidad@tiendasur.example. Respondemos en un máximo de 30 días.
## Cuenta y pedidos

### Cambiar la dirección
Puedes cambiar la dirección de entrega desde tu cuenta hasta que el pedido sale del almacén. Después hay que contactar con el transportista, y no podemos garantizar el cambio.

### Cancelar un pedido
Un pedido se puede cancelar sin coste hasta que se prepara, normalmente en las 2 horas siguientes a la compra. Pasado ese plazo se tramita como una devolución.

### Contraseña olvidada
Pulsa "He olvidado mi contraseña" y recibirás por correo un enlace válido durante 1 hora.

## Regalos

### Envoltorio y tarjeta
El envoltorio de regalo cuesta 2,50 EUR por pedido. La tarjeta dedicatoria es gratis hasta 150 caracteres y el albarán va sin precios.

### Cheque regalo
Los cheques regalo caducan a los 12 meses y no se reembolsan en efectivo.

## Cafeteras y reparaciones

### Reparación
Las cafeteras eléctricas en garantía se reparan en nuestro taller en 10 días laborables. Fuera de garantía, el presupuesto previo es gratuito.

### Descalcificación
Descalcifica la cafetera cada 2 meses con una solución de ácido cítrico; el vinagre daña las juntas.

## Envíos internacionales

### Países y plazos
Enviamos a Portugal, Francia e Italia en 4 a 6 días laborables, con un coste de 9,90 EUR. Fuera de la Unión Europea no enviamos.

## Pedidos para empresas

### Compras al por mayor
Los pedidos de más de 20 bolsas de café tienen un 15 % de descuento; escribe a empresas@tiendasur.example. Se factura a 30 días.

## Reclamaciones

### Cómo reclamar
Si no estás conforme, escribe a soporte: respondemos en 48 horas. En la tienda física hay hojas de reclamaciones.
"""

# Pares (pregunta del cliente, frase que contiene la respuesta). Las preguntas están escritas
# con otras palabras que el manual a propósito: así se mide la recuperación de verdad.
PREGUNTAS = [
    ('¿Cuánto tarda en llegar mi pedido a Canarias?', '5 a 8 días laborables'),
    ('¿A partir de qué importe no pago gastos de envío?', 'gratuito a partir de 39 EUR'),
    ('Necesito factura a nombre de mi empresa, ¿hasta cuándo puedo pedirla?', 'solo hasta 30 días tras la compra'),
    ('Me equivoqué en los datos de la factura, ¿la pueden corregir?', 'se emite una rectificativa'),
    ('¿Puedo devolver un paquete de café que ya abrí?', 'El café abierto no se admite'),
    ('¿Cuánto tardan en devolverme el dinero?', 'máximo de 10 días laborables'),
    ('¿Quién paga el transporte si devuelvo algo?', 'corren a cargo del cliente'),
    ('Me llegó una taza hecha pedazos, ¿qué hago?', 'Las tazas rotas en el transporte'),
    ('Llevo más de una semana esperando el paquete', 'no llega en 7 días laborables'),
    ('¿Qué días tostáis el café?', 'martes y jueves'),
    ('¿Cómo guardo el café para que no pierda aroma?', 'recipiente hermético'),
    ('Soy alérgico a los frutos secos, ¿el café puede llevarlos?', 'procesan frutos secos'),
    ('¿Cada cuánto me llega la suscripción?', 'cada 2 o 4 semanas'),
    ('Quiero darme de baja de la suscripción', 'pausar o cancelar desde tu cuenta'),
    ('¿Aceptáis Bizum?', 'Aceptamos tarjeta, Bizum'),
    ('¿Qué hace el código BIENVENIDA10?', 'BIENVENIDA10 da un 10 %'),
    ('¿Cuántos puntos necesito para conseguir 5 euros de descuento?', '500 puntos equivalen'),
    ('¿Cuántos años de garantía tienen las tazas?', '3 años de garantía legal'),
    ('¿Puedo meter el plato de cerámica en el lavavajillas?', 'uso en lavavajillas'),
    ('¿A qué hora cierra la tienda de Sevilla?', 'de 10:00 a 20:00'),
    ('¿Cómo puedo borrar mis datos?', 'acceso, rectificación o borrado'),
    ('¿Guardáis el número de mi tarjeta?', 'No almacenamos el número completo'),
    ('El seguimiento lleva días sin actualizarse', 'no cambia en 72 horas'),
    ('¿Puedo recoger el pedido en la tienda?', 'recogerse allí gratis'),
    ('Pagué por transferencia y no la habéis recibido, ¿se cancela?', '3 días o el pedido se cancela'),
    ('¿Se pueden juntar dos cupones en el mismo pedido?', 'no son acumulables'),
    ('¿Cuánto cuesta el envío urgente?', '8,50 EUR'),
    ('¿Cuánto tiempo dura el café una vez tostado?', 'seis semanas'),
    ('Me equivoqué de dirección al comprar, ¿puedo cambiarla?', 'cambiar la dirección de entrega desde tu cuenta'),
    ('Quiero anular lo que acabo de comprar', 'se puede cancelar sin coste'),
    ('¿Puedo usar vinagre para limpiar la cafetera?', 'el vinagre daña las juntas'),
    ('¿Hacéis envíos a Francia?', 'Portugal, Francia e Italia'),
    ('Si compro 30 bolsas para mi oficina, ¿hay descuento?', 'más de 20 bolsas'),
    ('¿Cuánto cuesta envolver para regalo?', '2,50 EUR'),
    ('¿Puedo cambiar un cheque regalo por dinero?', 'no se reembolsan en efectivo'),
    ('Mi cafetera eléctrica se estropeó, ¿cuánto tardan en arreglarla?', 'se reparan en nuestro taller'),
]

for _, evidencia in PREGUNTAS:
    assert evidencia in MANUAL, evidencia      # la evidencia debe existir en el manual
print(f"{len(MANUAL.split())} palabras en el manual · {len(PREGUNTAS)} preguntas de evaluación")

## 3.2 · Texto y embeddings sin descargas

**Decisión:** no descargamos ningún modelo. Un modelo de embeddings multilingüe decente ocupa de ~120 MB (versión pequeña) a varios cientos de MB y exige `torch`; es más de lo que conviene para un notebook que tiene que correr en cualquier portátil en minutos. Usamos en su lugar **TF-IDF de n-gramas de letras + SVD** (*Latent Semantic Analysis*): es un embedding de verdad (vectores densos de 32 dimensiones, coseno), se entrena en milisegundos y capta morfología (`devolver`/`devolución`). **No capta sinónimos que nunca coinciden** (`anular`/`cancelar`): eso es exactamente lo que enseñaremos a medir. Al final hay una celda de Colab para cambiarlo por un modelo preentrenado sin tocar el resto.

In [ ]:
def normalizar(texto: str) -> str:
    """Minúsculas y sin tildes: 'Devolución' y 'devolucion' pasan a ser la misma palabra."""
    t = unicodedata.normalize("NFD", texto.lower())
    return "".join(c for c in t if unicodedata.category(c) != "Mn")

PALABRAS_VACIAS = {normalizar(p) for p in """de la el en y a los las un una que es se por con para al del lo su o
no me mi si te tu como cuanto cuantos cuando qué hay ha han puedo puede pueden hacer""".split()}

def tokenizar(texto: str, raiz: int = 5) -> list[str]:
    """Palabras sin tildes ni palabras vacías, recortadas a las 5 primeras letras.
    Es un 'stemmer' tosco: devolver/devolución -> 'devol'. Basta para el español de este manual."""
    palabras = re.findall(r"[a-z0-9]+", normalizar(texto))
    return [p[:raiz] for p in palabras if p not in PALABRAS_VACIAS]

print(tokenizar("¿Cuánto tardan en devolverme el dinero de la devolución?"))

In [ ]:
def normalizar_filas(M: np.ndarray) -> np.ndarray:
    return M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)

class Embebedor:
    """Embeddings locales sin descargas: TF-IDF de n-gramas de letras + reducción SVD (LSA).

    - Los n-gramas de letras (3 a 5) hacen que 'devolver' y 'devolución' compartan trozos.
    - La SVD comprime a pocas dimensiones y junta palabras que aparecen en los mismos contextos.
    - Se ajusta con los chunks del propio manual: en producción se usa un modelo preentrenado.
    """
    def __init__(self, textos: list[str], dim: int = 32):
        self.tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), preprocessor=normalizar,
                                     sublinear_tf=True)
        X = self.tfidf.fit_transform(textos)
        self.dim = max(2, min(dim, X.shape[0] - 1, X.shape[1] - 1))
        self.svd = TruncatedSVD(n_components=self.dim, random_state=0).fit(X)

    def __call__(self, textos: list[str]) -> np.ndarray:
        return normalizar_filas(self.svd.transform(self.tfidf.transform(textos)))

emb = Embebedor([s for s in re.split(r"\n\n", MANUAL) if len(s) > 80])
v = emb(["¿Cuánto tarda la devolución del dinero?", "reembolso de mi pago",
         "El café se tuesta los martes", "plazo para devolver y recuperar el importe"])
print("dimensión del embedding:", v.shape[1])
print("coseno devolución~reembolso :", round(float(v[0] @ v[1]), 2))
print("coseno devolución~plazo     :", round(float(v[0] @ v[3]), 2))
print("coseno devolución~café      :", round(float(v[0] @ v[2]), 2))

**Lectura:** «plazo para devolver y recuperar el importe» queda claramente más cerca que el café (comparte raíces: *devol*), pero «reembolso de mi pago» apenas se distingue de él: el modelo no sabe que reembolso y devolución del dinero son lo mismo, porque en el manual nunca coinciden. Con un modelo preentrenado esa pareja quedaría cerca; aquí no. Esta limitación es la excusa perfecta para la búsqueda híbrida y el reranking, y por eso vamos a **medir** en vez de fiarnos de ejemplos sueltos.

## 3.3 · Chunking: fijo, con solape y por estructura

Tres estrategias sobre el mismo manual. Fíjate en el corte ingenuo: una ventana de 200 caracteres parte frases y separa el título de su contenido. La estrategia por estructura corta por subsecciones, **mete las migas de pan** (`Devoluciones - Reembolso: …`) dentro del texto y solo parte —con una frase de solape— lo que excede el máximo.

In [ ]:
def chunking_fijo(doc: str, tam: int = 200, solape: int = 0) -> list[dict]:
    """El método ingenuo: ventanas de `tam` caracteres, con `solape` caracteres compartidos."""
    cuerpo = re.sub(r"\s+", " ", re.sub(r"^#+ .*$", "", doc, flags=re.M)).strip()
    chunks, i = [], 0
    while i < len(cuerpo):
        chunks.append({"id": f"fijo-{len(chunks)}", "seccion": "", "texto": cuerpo[i:i + tam]})
        if i + tam >= len(cuerpo):
            break
        i += tam - solape
    return chunks

def _dividir_largo(texto: str, max_car: int, frases_solape: int = 1) -> list[str]:
    """Divide un texto largo por frases, agrupándolas hasta max_car y repitiendo la última frase."""
    frases = re.split(r"(?<=[.!?])\s+", texto.strip())
    trozos, actual = [], []
    for f in frases:
        if actual and len(" ".join(actual + [f])) > max_car:
            trozos.append(" ".join(actual))
            actual = actual[-frases_solape:] if frases_solape else []
        actual.append(f)
    trozos.append(" ".join(actual))
    return trozos

def chunking_estructura(doc: str, max_car: int = 600) -> list[dict]:
    """Un chunk por subsección (###). El título y la sección van DENTRO del texto (migas de pan),
    y las subsecciones muy largas se parten por frases con una frase de solape."""
    chunks, seccion = [], ""
    for bloque in re.split(r"\n(?=##+ )", doc):
        titulo, _, cuerpo = bloque.strip().partition("\n")
        if titulo.startswith("## "):
            seccion = titulo[3:]
        elif titulo.startswith("### "):
            sub = titulo[4:]
            for k, parte in enumerate(_dividir_largo(cuerpo, max_car)):
                sufijo = f" (parte {k + 1})" if k else ""
                chunks.append({"id": f"{seccion} > {sub}{sufijo}", "seccion": seccion,
                               "texto": f"{seccion} - {sub}: {parte}"})
    return chunks

c_fijo, c_solape, c_estr = chunking_fijo(MANUAL), chunking_fijo(MANUAL, 200, 40), chunking_estructura(MANUAL)
for nombre, cs in [("fijo 200", c_fijo), ("fijo 200 + solape 40", c_solape), ("por estructura", c_estr)]:
    print(f"{nombre:22s} {len(cs):3d} chunks · longitud media {np.mean([len(c['texto']) for c in cs]):.0f} caracteres")
print("\nEjemplo de corte ingenuo (fijo 200):")
print(" ", c_fijo[5]["texto"], "\n ", c_fijo[6]["texto"])
print("\nEjemplo por estructura:")
print(" ", c_estr[7]["id"], "->", c_estr[7]["texto"][:110], "…")

largo = "### Prueba\n" + " ".join(f"Esta es la frase número {i} de una sección muy larga." for i in range(1, 25))
print("\nSubsección larga ->", len(chunking_estructura("## S\n\n" + largo, 300)), "chunks con una frase de solape")

## 3.4 · La base vectorial: SQLite + `sqlite-vec`

Para guardar vectores y buscar los más cercanos hace falta una **base vectorial**. Tres niveles:
1. **NumPy a fuerza bruta** (el módulo anterior): perfecto hasta decenas de miles de chunks.
2. **Extensión de base de datos sin servidor** (aquí `sqlite-vec`: un solo archivo, `pip install`, ideal para prototipos y apps locales).
3. **Servicio de producción:** PostgreSQL + **pgvector**, OpenSearch/Elasticsearch, Qdrant, etc., con índices aproximados (HNSW) para millones de vectores, filtros, réplicas y permisos.

Construimos la clase `Indice` (con las tres formas de ordenar los chunks) y la primera es la consulta KNN a la base vectorial. Ya que estamos, añadimos BM25 (3.5) en la misma clase.

In [ ]:
class BM25:
    """Búsqueda léxica BM25 desde cero (la fórmula de OpenSearch/Elasticsearch/Lucene).
    puntúa = suma por palabra de la consulta de  idf · f·(k1+1) / (f + k1·(1 - b + b·len/len_media))."""
    def __init__(self, textos: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.docs = [tokenizar(t) for t in textos]
        self.N = len(self.docs)
        self.len_media = sum(len(d) for d in self.docs) / self.N
        df: dict[str, int] = {}
        for d in self.docs:
            for p in set(d):
                df[p] = df.get(p, 0) + 1
        self.idf = {p: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for p, n in df.items()}

    def puntuar(self, consulta: str) -> np.ndarray:
        s = np.zeros(self.N)
        for p in set(tokenizar(consulta)):
            if p not in self.idf:
                continue
            for i, d in enumerate(self.docs):
                f = d.count(p)
                if f:
                    s[i] += self.idf[p] * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * len(d) / self.len_media))
        return s

def rrf(listas: list[list[int]], k: int = 60, pesos: list[float] | None = None) -> list[int]:
    """Reciprocal Rank Fusion: cada lista aporta 1/(k + posición). Solo usa POSICIONES,
    así que no importa que BM25 y el coseno tengan escalas distintas."""
    pesos = pesos or [1.0] * len(listas)
    puntos: dict[int, float] = {}
    for lista, w in zip(listas, pesos):
        for pos, i in enumerate(lista):
            puntos[i] = puntos.get(i, 0.0) + w / (k + pos + 1)
    return sorted(puntos, key=lambda i: -puntos[i])

print("Ejemplo RRF -> BM25 dice [2,0,1], el vector dice [0,2,1]:", rrf([[2, 0, 1], [0, 2, 1]]))

In [ ]:
class Indice:
    """Todo lo que necesita un buscador: chunks + BM25 + embeddings + base vectorial sqlite-vec."""
    def __init__(self, chunks: list[dict], ruta_db: str | Path = ":memory:", dim: int = 32):
        self.chunks = chunks
        textos = [c["texto"] for c in chunks]
        self.bm25 = BM25(textos)
        self.emb = Embebedor(textos, dim)
        self.E = self.emb(textos)
        self.db = self._crear_db(ruta_db)

    def _crear_db(self, ruta_db):
        """Base vectorial real sin servidor: SQLite + extensión sqlite-vec (un solo archivo)."""
        db = sqlite3.connect(str(ruta_db))
        db.enable_load_extension(True)
        sqlite_vec.load(db)
        db.enable_load_extension(False)
        db.execute("DROP TABLE IF EXISTS chunks")
        db.execute("DROP TABLE IF EXISTS vec_chunks")
        db.execute("CREATE TABLE chunks (id INTEGER PRIMARY KEY, nombre TEXT, seccion TEXT, texto TEXT)")
        db.execute(f"CREATE VIRTUAL TABLE vec_chunks USING vec0(embedding float[{self.E.shape[1]}] distance_metric=cosine)")
        for i, (c, v) in enumerate(zip(self.chunks, self.E)):
            db.execute("INSERT INTO chunks VALUES (?,?,?,?)", (i, c["id"], c["seccion"], c["texto"]))
            db.execute("INSERT INTO vec_chunks(rowid, embedding) VALUES (?,?)",
                       (i, sqlite_vec.serialize_float32(v.astype(np.float32))))
        db.commit()
        return db

    # --- tres formas de ordenar TODOS los chunks para una consulta --------------------
    def orden_bm25(self, q: str) -> list[int]:
        return [int(i) for i in np.argsort(-self.bm25.puntuar(q), kind="stable")]

    def orden_vector(self, q: str, k: int | None = None) -> list[int]:
        """Consulta KNN a la base vectorial (sqlite-vec): devuelve ids ordenados por distancia coseno."""
        v = self.emb([q])[0].astype(np.float32)
        k = k or len(self.chunks)
        filas = self.db.execute("SELECT rowid, distance FROM vec_chunks WHERE embedding MATCH ? AND k = ? ORDER BY distance",
                                (sqlite_vec.serialize_float32(v), k)).fetchall()
        return [r[0] for r in filas]

    def orden_hibrido(self, q: str, k_rrf: int = 60, pesos=None) -> list[int]:
        return rrf([self.orden_bm25(q), self.orden_vector(q)], k=k_rrf, pesos=pesos)

    def coseno_max(self, q: str) -> float:
        return float(np.max(self.E @ self.emb([q])[0]))

idx = Indice(c_estr, CARPETA / "tiendasur.db")
print("filas en la base vectorial:", idx.db.execute("SELECT count(*) FROM vec_chunks").fetchone()[0],
      "· archivo:", (CARPETA / "tiendasur.db").name, f"({(CARPETA / 'tiendasur.db').stat().st_size // 1024} KB)")

q = "¿Cuánto tardan en devolverme el dinero?"
print("\nConsulta:", q)
for i in idx.orden_vector(q, 3):
    print(f"  vector -> [{idx.chunks[i]['id']}]")
for i in idx.orden_bm25(q)[:3]:
    print(f"  bm25   -> [{idx.chunks[i]['id']}]")
for i in idx.orden_hibrido(q)[:3]:
    print(f"  RRF    -> [{idx.chunks[i]['id']}]")

In [ ]:
# Comprobación: la base vectorial da el mismo orden que la búsqueda a fuerza bruta con NumPy.
q = "¿Cuánto tardan en devolverme el dinero?"
bruta = [int(i) for i in np.argsort(-(idx.E @ idx.emb([q])[0]), kind="stable")[:5]]
print("sqlite-vec:", idx.orden_vector(q, 5))
print("NumPy     :", bruta)
assert idx.orden_vector(q, 5) == bruta

# Metadatos: una consulta vectorial + filtro SQL sobre la sección (el JOIN lo hace SQLite)
v = sqlite_vec.serialize_float32(idx.emb([q])[0].astype(np.float32))
filas = idx.db.execute("""
    SELECT c.nombre, round(1 - v.distance, 3) AS coseno
    FROM (SELECT rowid, distance FROM vec_chunks WHERE embedding MATCH ? AND k = 10) v
    JOIN chunks c ON c.id = v.rowid
    WHERE c.seccion = 'Devoluciones'
    ORDER BY v.distance LIMIT 3""", (v,)).fetchall()
print("\nSolo en la sección «Devoluciones»:", filas)

**Lectura:** `sqlite-vec` devuelve exactamente el orden de la fuerza bruta (es búsqueda exacta; los índices aproximados de pgvector/HNSW cambian algo de exactitud por velocidad). Lo valioso es el **diseño**: tabla de metadatos + tabla de vectores + SQL con filtros. Este es el mismo diseño en PostgreSQL:

In [ ]:
# El MISMO diseño en producción con PostgreSQL + pgvector (no se ejecuta aquí: es el SQL equivalente).
SQL_PGVECTOR = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE chunks (
    id         bigserial PRIMARY KEY,
    nombre     text,
    seccion    text,
    texto      text,
    embedding  vector(384),                      -- la dimensión de TU modelo de embeddings
    tsv        tsvector GENERATED ALWAYS AS (to_tsvector('spanish', texto)) STORED
);

-- Índices: HNSW para vecinos aproximados (rápido a millones de filas), GIN para texto
CREATE INDEX ON chunks USING hnsw (embedding vector_cosine_ops);
CREATE INDEX ON chunks USING gin (tsv);

-- Búsqueda vectorial: <=> es la distancia coseno ($1 = embedding de la consulta)
SELECT id, nombre, 1 - (embedding <=> $1) AS coseno
FROM chunks WHERE seccion = 'Devoluciones'       -- filtro por metadatos
ORDER BY embedding <=> $1 LIMIT 10;

-- Búsqueda híbrida con RRF en una sola consulta ($2 = texto de la consulta)
WITH vec AS (
    SELECT id, row_number() OVER (ORDER BY embedding <=> $1) AS r
    FROM chunks ORDER BY embedding <=> $1 LIMIT 20),
lex AS (
    SELECT id, row_number() OVER (ORDER BY ts_rank_cd(tsv, q) DESC) AS r
    FROM chunks, websearch_to_tsquery('spanish', $2) q WHERE tsv @@ q LIMIT 20)
SELECT coalesce(v.id, l.id) AS id,
       coalesce(1.0 / (60 + v.r), 0) + coalesce(1.0 / (60 + l.r), 0) AS rrf
FROM vec v FULL OUTER JOIN lex l ON v.id = l.id
ORDER BY rrf DESC LIMIT 5;
"""
print(SQL_PGVECTOR)
# Nota: ts_rank_cd NO es BM25. Para BM25 real en Postgres existen extensiones (p. ej. ParadeDB pg_search);
# en OpenSearch/Elasticsearch el BM25 viene de serie. Comprueba qué ofrece tu proveedor.

## 3.5 · Búsqueda híbrida: BM25 + vectorial + RRF

- **BM25** (léxica) acierta con **palabras exactas**: códigos (`BIENVENIDA10`), nombres, cifras. Falla con sinónimos.
- **Vectorial** acierta con **paráfrasis y morfología**. Falla con identificadores raros y a veces «se inventa» parecidos.
- **RRF** (*Reciprocal Rank Fusion*) las combina usando solo las **posiciones** de cada lista, sin tener que calibrar escalas distintas. Es el método por defecto en casi todos los motores híbridos.

Ya tienes `BM25`, `rrf` e `Indice.orden_hibrido` más arriba. Dos comprobaciones: nuestro BM25 frente al de SQLite (FTS5), y la tabla de resultados.

In [ ]:
# BM25 "de fábrica" dentro de SQLite: FTS5. Comparamos con nuestro BM25 casero.
idx.db.execute("DROP TABLE IF EXISTS fts")
idx.db.execute("CREATE VIRTUAL TABLE fts USING fts5(texto, tokenize='unicode61 remove_diacritics 2')")
idx.db.executemany("INSERT INTO fts(rowid, texto) VALUES (?,?)", [(i, c["texto"]) for i, c in enumerate(idx.chunks)])

def orden_fts5(q: str) -> list[int]:
    """FTS5 con prefijos de 5 letras ('devol'*) = el mismo truco de raíz que nuestro tokenizador."""
    terminos = " OR ".join(f'"{t}"*' for t in sorted(set(tokenizar(q))))
    filas = idx.db.execute("SELECT rowid FROM fts WHERE fts MATCH ? ORDER BY bm25(fts)", (terminos,)).fetchall()
    vistos = [r[0] for r in filas]
    return vistos + [i for i in range(len(idx.chunks)) if i not in vistos]   # completa con el resto

print([idx.chunks[i]["id"] for i in orden_fts5("¿Cuánto tardan en devolverme el dinero?")[:3]])

## 3.6 · Evaluar la recuperación: acierto@k y MRR

- **acierto@k:** ¿hay un chunk correcto entre los k primeros? Es lo que le importa al generador (le pasas k chunks).
- **MRR:** el promedio de 1/posición del primer acierto; premia que el correcto vaya **el primero**.

Comparamos 3 chunkings × 3 variantes de búsqueda.

In [ ]:
def posicion_relevante(orden: list[int], chunks: list[dict], evidencia: str) -> int | None:
    """Posición (empezando en 1) del primer chunk que CONTIENE la frase de la respuesta."""
    for pos, i in enumerate(orden, start=1):
        if evidencia in chunks[i]["texto"]:
            return pos
    return None

def evaluar(orden_fn, chunks: list[dict], preguntas=PREGUNTAS) -> dict:
    """acierto@k = % de preguntas con un chunk correcto entre los k primeros.
    MRR = media de 1/posición del primer acierto (1.0 = siempre el primero)."""
    pos = [posicion_relevante(orden_fn(q), chunks, ev) for q, ev in preguntas]
    n = len(preguntas)
    return {"acierto@1": sum(p == 1 for p in pos) / n,
            "acierto@3": sum(p is not None and p <= 3 for p in pos) / n,
            "acierto@5": sum(p is not None and p <= 5 for p in pos) / n,
            "MRR": sum(1 / p for p in pos if p) / n}

filas = []
indices = {"fijo 200": Indice(c_fijo), "fijo 200 + solape 40": Indice(c_solape), "por estructura": idx}
for nombre_chunking, ix in indices.items():
    for variante, fn in [("BM25", ix.orden_bm25), ("vectorial", ix.orden_vector), ("híbrida RRF", ix.orden_hibrido)]:
        filas.append({"chunking": nombre_chunking, "variante": variante, "chunks": len(ix.chunks),
                      **evaluar(fn, ix.chunks)})
tabla = pd.DataFrame(filas).set_index(["chunking", "variante"]).round(2)
print(tabla.to_string())

**Cómo leerlo (compruébalo con TU salida, no con la mía):**
1. Mira primero el efecto del **chunking** dentro de una misma variante: es la palanca que más suele mover las métricas.
2. Después, ¿la híbrida iguala o supera a la mejor de las otras dos en **cada** fila? No tiene por qué ganar siempre: con 36 preguntas una sola pregunta mueve el MRR unos 0,03.
3. Mira el detalle por pregunta: BM25 gana en códigos y cifras; el vector gana donde cambian las palabras pero comparten raíz.

In [ ]:
# ¿Dónde falla cada variante con el mejor chunking? (posición del chunk correcto; '-' = fuera de la lista)
filas = []
for q, ev in PREGUNTAS:
    fila = {"pregunta": q[:52]}
    for variante, fn in [("BM25", idx.orden_bm25), ("vector", idx.orden_vector), ("híbrida", idx.orden_hibrido)]:
        fila[variante] = posicion_relevante(fn(q), idx.chunks, ev) or "-"
    filas.append(fila)
detalle = pd.DataFrame(filas)
print("Preguntas donde alguna variante NO acierta en el primer puesto:")
print(detalle[(detalle[["BM25", "vector", "híbrida"]] != 1).any(axis=1)].to_string(index=False))

## 3.7 · Reranking

La búsqueda recupera **candidatos** rápido (barato, recall alto); el **reranker** reordena solo los primeros N con un modelo que ve consulta y fragmento **juntos** (más preciso, más caro). Tres opciones:

| Reranker | Cuándo | Coste |
|---|---|---|
| Léxico casero (aquí) | sin clave ni modelo; solo ve palabras compartidas | casi cero |
| *Cross-encoder* pequeño | producción propia; comprueba el tamaño (algunos < 200 MB, los multilingües suelen pesar más) | CPU/GPU local |
| LLM (Claude) con salida estructurada | pocos candidatos, calidad alta | una llamada por consulta |

Medimos las que podemos ejecutar.

In [ ]:
def rerank_lexico(consulta: str, candidatos: list[int], ix: Indice) -> list[int]:
    """Reranker léxico SIN clave: reordena los candidatos por (a) cuánta idf de la consulta cubren
    y (b) cuántos pares de palabras seguidas coinciden. Es barato, pero solo ve palabras compartidas."""
    q = tokenizar(consulta)
    q_pares = set(zip(q, q[1:]))
    total_idf = sum(ix.bm25.idf.get(p, 0) for p in set(q)) + 1e-9
    puntos = []
    for i in candidatos:
        d = ix.bm25.docs[i]
        cobertura = sum(ix.bm25.idf.get(p, 0) for p in set(q) if p in d) / total_idf
        pares = len(q_pares & set(zip(d, d[1:])))
        puntos.append((cobertura + 0.2 * pares, i))
    return [i for _, i in sorted(puntos, key=lambda t: -t[0])]

from pydantic import BaseModel

class OrdenFragmentos(BaseModel):
    ids_ordenados: list[int]     # de más a menos útil para responder

MODELO_RERANK = "claude-haiku-5-5"      # el reranker se llama en CADA consulta: modelo barato y rápido
_CACHE_RERANK: dict[tuple, list[int]] = {}

def rerank_claude(consulta: str, candidatos: list[int], ix: Indice) -> list[int]:
    """[API] Reranking con un LLM: ve la consulta y todos los candidatos juntos (como un cross-encoder)."""
    clave = (consulta, tuple(candidatos))
    if clave in _CACHE_RERANK:
        return _CACHE_RERANK[clave]
    import anthropic
    fragmentos = "\n\n".join(f"[{i}] {ix.chunks[i]['texto']}" for i in candidatos)
    resp = anthropic.Anthropic().messages.parse(
        model=MODELO_RERANK, max_tokens=4000, output_config={"effort": "low"},
        messages=[{"role": "user", "content":
                   f"Ordena los fragmentos de más a menos útil para responder la consulta del cliente. "
                   f"Devuelve los ids de TODOS, sin repetir.\n\nConsulta: {consulta}\n\n{fragmentos}"}],
        output_format=OrdenFragmentos)
    orden = [i for i in resp.parsed_output.ids_ordenados if i in candidatos]
    orden += [i for i in candidatos if i not in orden]        # un LLM puede olvidar alguno: lo reponemos
    _CACHE_RERANK[clave] = orden
    return orden

N_CAND = 8      # el reranker solo reordena los N primeros de la búsqueda híbrida

def con_rerank(reranker):
    def orden(q):
        base = idx.orden_hibrido(q)
        return reranker(q, base[:N_CAND], idx) + base[N_CAND:]
    return orden

filas = [{"variante": "híbrida RRF (sin rerank)", **evaluar(idx.orden_hibrido, idx.chunks)},
         {"variante": "híbrida + rerank léxico", **evaluar(con_rerank(rerank_lexico), idx.chunks)}]
if TIENE_CLAVE:
    filas.append({"variante": "híbrida + rerank Claude", **evaluar(con_rerank(rerank_claude), idx.chunks)})
else:
    print("[SALTADO: sin clave — el reranking con Claude se evalúa solo con ANTHROPIC_API_KEY]")
print(pd.DataFrame(filas).set_index("variante").round(2).to_string())

**Lectura:** un reranker **solo puede reordenar lo que le das**: si el chunk correcto no está entre los N candidatos, no lo salva (por eso se pide recall alto a la búsqueda). El reranker léxico apenas aporta porque usa la misma señal (palabras compartidas) que BM25: no arregla el *desfase de vocabulario* («anular» ≠ «cancelar»). Un cross-encoder o un LLM sí entienden que son lo mismo; con clave, mira la tercera fila.

In [ ]:
# [COLAB / GPU opcional] Cross-encoder multilingüe abierto. Pesa varios cientos de MB, por eso NO se ejecuta aquí.
# %pip install -q sentence-transformers
# from sentence_transformers import CrossEncoder
# ce = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")     # comprueba el nombre vigente en el Hub
# def rerank_cross_encoder(consulta, candidatos, ix):
#     pares = [(consulta, ix.chunks[i]["texto"]) for i in candidatos]
#     puntos = ce.predict(pares)
#     return [i for _, i in sorted(zip(puntos, candidatos), reverse=True)]
#
# Y para sustituir nuestros embeddings por unos preentrenados, solo cambia la clase Embebedor:
# from sentence_transformers import SentenceTransformer
# modelo = SentenceTransformer("intfloat/multilingual-e5-small")      # unos 120 MB; ver nombre vigente
# class Embebedor:
#     def __init__(self, textos, dim=None): pass
#     def __call__(self, textos): return modelo.encode(["query: " + t for t in textos], normalize_embeddings=True)
# (e5 pide el prefijo "query: " en consultas y "passage: " en fragmentos; lee la ficha del modelo.)
print("Celda solo informativa (Colab).")

## 3.8 · Generación con contexto citado (y el «no lo sé»)

La generación ahora usa la **búsqueda híbrida**. Añadimos algo que en el módulo anterior faltaba: un **umbral de relevancia**. Si ni el mejor chunk se parece a la consulta, no se llama al modelo y se responde «no lo sé» (más barato y más seguro que confiar solo en el prompt).

In [ ]:
SYSTEM_RAG = textwrap.dedent("""
Eres el asistente de pedidos de TiendaSur.

Responde SOLO con la información de los fragmentos entre <docs>.
- Cita el fragmento con [id del chunk].
- Si los fragmentos no contestan la pregunta, di exactamente: "no lo sé".
""")

def buscar(consulta: str, top_k: int = 3) -> list[dict]:
    """Recuperación de producción de este módulo: búsqueda híbrida (BM25 + vector + RRF)."""
    orden = idx.orden_hibrido(consulta)[:top_k]
    return [{"id": idx.chunks[i]["id"], "texto": idx.chunks[i]["texto"]} for i in orden]

# ¿Hace falta un umbral para el "no lo sé"? Miramos el mejor coseno de preguntas con y sin respuesta.
SIN_RESPUESTA = ["¿Cuál es la capital de Francia?", "¿Venden té matcha en polvo?", "¿Quién ganó el Mundial de 2010?"]
# (la primera es una trampa a propósito: contiene «Francia», que SÍ aparece en el manual de envíos)
con = [idx.coseno_max(q) for q, _ in PREGUNTAS]
sin = [idx.coseno_max(q) for q in SIN_RESPUESTA]
print(f"coseno máximo con respuesta: mín {min(con):.2f} · mediana {np.median(con):.2f}")
print(f"coseno máximo SIN respuesta: {[round(s, 2) for s in sin]}")
UMBRAL = 0.55      # justo por debajo de la peor pregunta con respuesta; sobreajustado a estas pocas preguntas
print("umbral elegido:", UMBRAL, "(las preguntas trampa con palabras del manual lo superan: de ahí la 2.ª defensa, el prompt)")

In [ ]:
def responder_rag(consulta: str, top_k: int = 3) -> str:
    # Filtro previo: si ni el mejor chunk se parece a la consulta, ni siquiera llamamos al modelo.
    if idx.coseno_max(consulta) < UMBRAL:
        respuesta = "no lo sé"
        print(f"Q: {consulta}\nDocs: (ninguno supera el umbral)\nR: {respuesta}\n")
        return respuesta
    docs = buscar(consulta, top_k)
    contexto = "\n\n".join(f'<docs id="{d["id"]}">{d["texto"]}</docs>' for d in docs)
    prompt = f"{contexto}\n\nConsulta: {consulta}"
    if TIENE_CLAVE:
        import anthropic
        msg = anthropic.Anthropic().messages.create(
            model=MODELO, system=SYSTEM_RAG, max_tokens=4000, output_config={"effort": "low"},
            messages=[{"role": "user", "content": prompt}])
        respuesta = next(b.text for b in msg.content if b.type == "text")   # puede haber un bloque thinking antes
    else:
        respuesta = "\n".join(f"- [{d['id']}] {d['texto'][:90]}…" for d in docs[:2]) + "\n[SALTADO: sin clave — respuesta mock con los chunks]"
    print(f"Q: {consulta}\nDocs: {[d['id'] for d in docs]}\nR: {respuesta}\n")
    return respuesta

for q in ["Quiero anular lo que acabo de comprar", "¿Puedo devolver un café que ya abrí?", SIN_RESPUESTA[1], SIN_RESPUESTA[0]]:
    responder_rag(q)

## 3.9 · Agente: herramientas + bucle de decisión

El agente es el mismo de antes: su herramienta `buscar_docs` ahora usa la búsqueda híbrida (mejorar el RAG mejora al agente gratis). Repasa dos reglas del bucle: **una `tool_result` por cada `tool_use`** en un único mensaje, y **límite de vueltas**.

In [ ]:
TICKETS = []

def crear_ticket(tema: str, pedido_id: int | None = None) -> dict:
    """Crea un ticket de soporte."""
    t = {"id": len(TICKETS) + 1, "tema": tema, "pedido": pedido_id}
    TICKETS.append(t)
    return t

def get_pedido(pedido_id: int) -> dict:
    PEDIDOS = {1042: {"producto": "Tazas TiendaSur", "estado": "entregado", "fecha": "2026-02-06"}}
    return PEDIDOS.get(pedido_id, {"error": f"pedido {pedido_id} no existe"})

TOOLS = {
    "buscar_docs":  {"desc": "Busca en el manual de atención al cliente de TiendaSur (búsqueda híbrida) y devuelve fragmentos.",
                     "fn": lambda consulta: buscar(consulta, 3)},
    "get_pedido":   {"desc": "Estado de un pedido por id.",
                     "fn": lambda pedido_id: get_pedido(pedido_id)},
    "crear_ticket": {"desc": "Crea un ticket de soporte por tema; opcionalmente con id de pedido.",
                     "fn": crear_ticket},
}

AGENTE_SYSTEM = textwrap.dedent(SYSTEM_RAG + """

Además tienes herramientas útiles para resolver incidencias:
- `buscar_docs(consulta)`: busca en el manual.
- `get_pedido(pedido_id)`: estado de un pedido.
- `crear_ticket(tema, pedido_id)`: registra la incidencia y devuelve su id.

Plan: consulta el manual o el pedido que necesites, y para problemas reales
crea el ticket y ciérrate respondiendo el id.""")

TOOL_SCHEMAS = [
    {"name": "buscar_docs", "description": TOOLS["buscar_docs"]["desc"],
     "input_schema": {"type": "object", "properties": {"consulta": {"type": "string"}},
                      "required": ["consulta"]}},
    {"name": "get_pedido", "description": TOOLS["get_pedido"]["desc"],
     "input_schema": {"type": "object", "properties": {"pedido_id": {"type": "integer"}},
                      "required": ["pedido_id"]}},
    {"name": "crear_ticket", "description": TOOLS["crear_ticket"]["desc"],
     "input_schema": {"type": "object", "properties": {"tema": {"type": "string"},
                                                       "pedido_id": {"type": "integer"}},
                      "required": ["tema"]}},
]

def ejecutar(nombre: str, args: dict):
    return TOOLS[nombre]["fn"](**args)

In [ ]:
def agente(tarea: str, max_vueltas: int = 5):
    """[API] Bucle del agente: decide → usa tools → responde. Sin API: mock del plan."""
    if not TIENE_CLAVE:
        print("[SALTADA: sin clave — flujo mock del plan del agente]")
        docs = buscar("paquete dañado")
        print("1) el agente consultaría buscar_docs('paquete dañado') →", [d["id"] for d in docs])
        print("2) crear_ticket(tema='paquete dañado') →", crear_ticket("paquete dañado (mock)"))
        return
    import anthropic
    c = anthropic.Anthropic()
    mensajes = [{"role": "user", "content": tarea}]
    for vuelta in range(max_vueltas):
        resp = c.messages.create(model=MODELO, system=AGENTE_SYSTEM,
                                 max_tokens=8000, tools=TOOL_SCHEMAS, messages=mensajes)
        if resp.stop_reason != "tool_use":
            final = next((b.text for b in resp.content if b.type == "text"), "")
            print("→ respuesta final:", final[:300])
            return
        resultados = []   # una tool_result por CADA tool_use (llamadas paralelas), en un solo mensaje
        for bloque in (b for b in resp.content if b.type == "tool_use"):
            print(f"vuelta {vuelta+1}: {bloque.name}({bloque.input})")
            resultado = ejecutar(bloque.name, bloque.input)
            print("   resultado:", str(resultado)[:150])
            resultados.append({"type": "tool_result", "tool_use_id": bloque.id,
                               "content": json.dumps(resultado)})
        mensajes.append({"role": "assistant", "content": resp.content})
        mensajes.append({"role": "user", "content": resultados})
    print("(límite de vueltas: el agente no terminó — revisa la herramienta o el prompt)")

print("— Incidencia —")
agente("Mi pedido 1042 llegó con una taza rota. ¿Se puede devolver? Registra el problema")

## 3.10 · Cuándo NO agente (el anti-teatro) y una nota sobre MCP

El router determinista resuelve la mayoría de las incidencias con cero llamadas de herramienta del modelo. El agente se justifica cuando el orden de las acciones no se puede anticipar.

In [ ]:
def soporte_router(consulta: str) -> str:
    """Router determinista: la mitad de los 'agentes' necesitan solo esto."""
    docs = buscar(consulta, top_k=1)
    if "dañados o perdidos" in docs[0]["id"]:
        t = crear_ticket(tema=f"incidencia: {consulta}")
        return f"He abierto el ticket #{t['id']}. Un humano te responderá."
    return docs[0]["texto"]

print(soporte_router("el paquete me llegó roto"))
print(soporte_router("¿cuánto cuesta el envío urgente?"))

**Lectura:** este router abre un ticket si el mejor fragmento es de «Productos dañados o perdidos» y, si no, responde con el fragmento.

> **Nota sobre MCP.** Las herramientas de este agente (`buscar_docs`, `get_pedido`, `crear_ticket`) están definidas *a mano* dentro de tu código. El **Model Context Protocol (MCP)** es el estándar para exponer herramientas y datos desde un servidor reutilizable (el mismo `buscar_docs` serviría a Claude Desktop, a un IDE y a tu app). Lo trabajamos a fondo en el módulo **AI10**; aquí solo retén la idea: la lógica de recuperación que acabas de construir es lo que se publicaría detrás de un servidor MCP.

---

## Resumen del notebook

- **Chunking por estructura** (con migas de pan y solape en las partes largas) > ventanas fijas.
- **Base vectorial sin servidor:** SQLite + `sqlite-vec`; en producción, pgvector/OpenSearch con el mismo diseño.
- **Híbrida (BM25 + vector + RRF)** y **reranking**: se deciden **midiendo** (acierto@k, MRR), no por intuición.
- **Agente** = bucle de herramientas con límite de vueltas; el router determinista cubre la mayoría de casos.

---

## Ejercicios (5) — entregables

**1.** *Chunking.* Barre el tamaño de ventana (`150`, `300`, `600` caracteres) con solape 0 % y 20 %. Guarda el MRR de la búsqueda híbrida en una tabla. ¿Alcanza algún chunking fijo al de estructura? ¿Qué sesgo tiene la métrica con chunks grandes?

**2.** *Reciprocal Rank Fusion.* Prueba `k_rrf` en `(1, 10, 60, 200)` y los pesos `[2, 1]` y `[1, 2]` (BM25 y vector). ¿Cambia el MRR? ¿Con qué conjunto de datos te fiarías de la diferencia?

**3.** *Reescritura de consulta.* Crea un diccionario de sinónimos del cliente (`anular → cancelar`, `cupón → código`, `corregir → rectificativa`…), expande la consulta antes de BM25 y mide la ganancia. ¿Qué preguntas arregla y cuáles no?

**4.** *Filtros por metadatos.* Con un router de una línea que elige la sección por palabras clave (`factura` → `Facturas`…), filtra la búsqueda vectorial en SQL. ¿Mejora el acierto@1? ¿Qué riesgo tiene filtrar mal?

**5.** *Router frente a agente.* Reescribe el caso «quiero cambiar la dirección de un pedido» como **router determinista** (dos `if` y dos respuestas). Escribe en 3 líneas por qué es mejor que un agente aquí.

> Recuerda: a la **búsqueda** se le exige lo mismo que a un modelo: una métrica antes y después de cada cambio.


In [ ]:
# Ejercicio 1 · barrido de chunking fijo
filas = []
for tam in (150, 300, 600):
    for solape_pct in (0, 20):
        cs = chunking_fijo(MANUAL, tam, int(tam * solape_pct / 100))
        ix = Indice(cs)
        filas.append({"tamaño": tam, "solape %": solape_pct, "chunks": len(cs), "car. medios": int(np.mean([len(c["texto"]) for c in cs])), "MRR híbrida": round(evaluar(ix.orden_hibrido, cs)["MRR"], 3)})
filas.append({"tamaño": "estructura", "solape %": "-", "chunks": len(c_estr), "car. medios": int(np.mean([len(c["texto"]) for c in c_estr])), "MRR híbrida": round(evaluar(idx.orden_hibrido, c_estr)["MRR"], 3)})
print(pd.DataFrame(filas).to_string(index=False))
# Trampa de la métrica: un chunk de 600 caracteres abarca ~3 subsecciones y "contiene" la respuesta
# con más facilidad, así que su MRR sube. Pero entrega a la vez 3 temas distintos al LLM (más ruido, más
# tokens, más coste). Por eso se mira también la longitud media y, sobre todo, la calidad de la
# respuesta final, no solo la recuperación.

In [ ]:
# Ejercicio 2 · RRF: k y pesos
filas = []
for k in (1, 10, 60, 200):
    for pesos in (None, [2, 1], [1, 2]):
        r = evaluar(lambda q: idx.orden_hibrido(q, k_rrf=k, pesos=pesos), idx.chunks)
        filas.append({"k": k, "pesos (BM25, vector)": pesos or [1, 1], "acierto@1": round(r["acierto@1"], 2), "MRR": round(r["MRR"], 3)})
print(pd.DataFrame(filas).to_string(index=False))
# Con 36 preguntas, una sola pregunta mueve el MRR ~0,03: diferencias menores son ruido.
# Para fiarte necesitas cientos de preguntas reales (logs) y mirar el intervalo de confianza.

In [ ]:
# Ejercicio 3 · reescritura de consulta con sinónimos
SINONIMOS = {"anular": "cancelar", "cupones": "códigos", "cupón": "código", "juntar": "acumulables",
             "corregir": "rectificativa", "transporte": "envío", "baja": "cancelar", "arreglarla": "reparación",
             "pedazos": "rotas", "roto": "dañado"}

def expandir(q: str) -> str:
    extra = [SINONIMOS[p] for p in re.findall(r"\w+", q.lower()) if p in SINONIMOS]
    return q + " " + " ".join(extra)

base = evaluar(idx.orden_bm25, idx.chunks)
exp = evaluar(lambda q: idx.orden_bm25(expandir(q)), idx.chunks)
print("BM25 sin expandir :", {k: round(v, 2) for k, v in base.items()})
print("BM25 con sinónimos:", {k: round(v, 2) for k, v in exp.items()})
# Ojo: el diccionario está hecho mirando estas preguntas -> la ganancia está INFLADA (fuga de evaluación).
# En la práctica los sinónimos se sacan de los logs de búsqueda reales, o se deja al LLM reescribir la consulta.

In [ ]:
# Ejercicio 4 · filtro por sección con metadatos
RUTAS = {"factura": "Facturas", "devol": "Devoluciones", "envío": "Envíos", "envio": "Envíos",
         "suscrip": "Suscripciones", "garant": "Menaje", "tienda": "Tienda física"}

def seccion_de(q: str):
    for clave, sec in RUTAS.items():
        if clave in normalizar(q) or clave in q.lower():
            return sec
    return None

def orden_filtrado(q: str) -> list[int]:
    base = idx.orden_hibrido(q)
    sec = seccion_de(q)
    if not sec:
        return base
    dentro = [i for i in base if idx.chunks[i]["seccion"] == sec]
    return dentro + [i for i in base if i not in dentro]      # filtrar = subir, no borrar el resto

print("sin filtro :", {k: round(v, 2) for k, v in evaluar(idx.orden_hibrido, idx.chunks).items()})
print("con filtro :", {k: round(v, 2) for k, v in evaluar(orden_filtrado, idx.chunks).items()})
# Riesgo: si el router se equivoca de sección, el chunk correcto desaparece de los primeros puestos.
# Por eso aquí el filtro "sube" en vez de "borrar".

In [ ]:
# Ejercicio 5 · cambiar la dirección: router determinista
def router_direccion(consulta: str) -> str:
    if "direcci" in normalizar(consulta):
        return buscar("cambiar la dirección de entrega", 1)[0]["texto"]
    return "Esto no es un cambio de dirección: te paso con una persona."

print(router_direccion("quiero cambiar la dirección de mi pedido 1042"))
print(router_direccion("¿qué horario tienen?"))
# Por qué es mejor que un agente: (1) el flujo es siempre el mismo -> no hace falta planificar;
# (2) cero llamadas al modelo: más barato, rápido y reproducible; (3) se puede testear con asserts.